# HEATMAP VISUALIZATION V2

## MSCOCO

In [ ]:
import math
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    v_measure_score,
)

sys.path.append(os.path.abspath(".."))
from dataset.mscoco.mscoco_dataloader_with_imagenet_labels import (
    MSCOCOEmbeddingsDatasetWithImageNetLabels,
    mscoco_imagenet_collate_fn,
)

from analysis.modality_gap import compute_gap
from metrics.clustering import clustering_metrics_two_modalities_mscoco_imagenet_labels
from metrics.retrieval import compute_retrieval
from subspace_alignment.subspace_alignment import (
    analyze_subspace_dimensions,
    apply_subspace_alignment,
    fit_subspace_alignment,
)

# ── User Configuration: edit this block when you change the model/dataset setup ──
DATASET_ROOT = Path("/mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco")
DATASET_NAME = "mscoco_imagenet_labels"
PREFERRED_CUDA_INDEX = 1

# CLIP_MODEL = "ViT-B-16"
# CLIP_PRETRAINED = "laion2b_s34b_b88k"
# ------------------------
# CLIP_MODEL = "ViT-bigG-14"
# CLIP_PRETRAINED = "laion2b_s39b_b160k"
# ------------------------
CLIP_MODEL = "ViT-L-16-SigLIP2-256"
CLIP_PRETRAINED = "webli"
MODEL_NAME = f"{CLIP_MODEL}___{CLIP_PRETRAINED}"

TRAIN_EMBED_SPLIT = "train2017"
TEST_EMBED_SPLIT = "val2017"
PRECOMPUTED_SUFFIX = "clip_imagenet"

BATCH_SIZE = 256
N_FIT_SAMPLES = 100_000
GAP_MAX_SAMPLES = 100_000
D_SUB = 512  # If None, use the embedding dimension inferred from the selected model.

PROJECT_ROOT = Path("/mnt/media/emanuele/few_dimensions")
HEATMAP_EXPORT_ROOT = PROJECT_ROOT / "figures_paper" / "heatmaps"
SAMPLE_IMAGE_FILENAMES = [
    "000000022892.jpg",  # cat dog plant
    "000000001761.jpg",  # planes and bridge
    "000000007281.jpg",  # man and horses
    "000000000885.jpg",  # tennis
    "000000000394.jpg",  # dog and frisbee
    "000000001025.jpg",  # dog sitting in a heart on a beach by ocean
    "000000001098.jpg",  # dog sitting in a heart on a beach by ocean
    "000000001108.jpg",  # man on the skateboard
    "000000001138.jpg",  # living room
    "000000000589.jpg",  # guy on the grass
    "000000000502.jpg",  # bear
]

# Patch-token count depends on the selected model and input resolution.
# We infer it dynamically later from spatial_tokens.shape[1], so there is no hardcoded value here.

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 123
seed = SEED
g = torch.Generator().manual_seed(seed)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    if torch.cuda.device_count() <= PREFERRED_CUDA_INDEX:
        raise RuntimeError(
            f"Requested cuda:{PREFERRED_CUDA_INDEX}, but only {torch.cuda.device_count()} CUDA device(s) are available."
        )
    device = f"cuda:{PREFERRED_CUDA_INDEX}"
else:
    device = "cpu"

dataset_name = DATASET_NAME

def build_precomputed_dir(split_name: str) -> str:
    return str(DATASET_ROOT / MODEL_NAME / f"precomputed_{split_name}_{PRECOMPUTED_SUFFIX}")

precomputed_dir = build_precomputed_dir(TRAIN_EMBED_SPLIT)
precomputed_dir_test = build_precomputed_dir(TEST_EMBED_SPLIT)

### Filtering procedure
As done in the few_dimension notebook

In [9]:
ds_train = MSCOCOEmbeddingsDatasetWithImageNetLabels(
    precomputed_dir,
    split_name="train_shard",
    return_label_name=False,
)

ds_test = MSCOCOEmbeddingsDatasetWithImageNetLabels(
    precomputed_dir_test,
    split_name="val_shard",
    return_label_name=False,
)


def _count_labels(dataset):
    counts = {}
    for i in range(len(dataset)):
        _, _, y = dataset[i]
        y = int(y.item()) if torch.is_tensor(y) else int(y)
        counts[y] = counts.get(y, 0) + 1
    return counts


train_counts = _count_labels(ds_train)
test_counts = _count_labels(ds_test)

test_classes = set(test_counts.keys())
keep_classes = {c for c in test_classes if train_counts.get(c, 0) >= 10}

train_indices = [
    i for i in range(len(ds_train))
    if int(ds_train[i][2].item()) in keep_classes
]

test_indices = [
    i for i in range(len(ds_test))
    if int(ds_test[i][2].item()) in keep_classes
]

filtered_train = torch.utils.data.Subset(ds_train, train_indices)
filtered_test = torch.utils.data.Subset(ds_test, test_indices)


def _count_labels_subset(subset):
    counts = {}
    for i in range(len(subset)):
        _, _, y = subset[i]
        y = int(y.item()) if torch.is_tensor(y) else int(y)
        counts[y] = counts.get(y, 0) + 1
    return counts


filtered_train_counts = _count_labels_subset(filtered_train)
filtered_test_counts = _count_labels_subset(filtered_test)

train_classes = set(filtered_train_counts.keys())
test_classes = set(filtered_test_counts.keys())

print(f"Kept classes (train >= 10 & in test): {len(keep_classes)}")
print(f"Train classes after filter: {len(train_classes)}")
print(f"Test classes after filter: {len(test_classes)}")
print(f"Train samples after filter: {len(filtered_train)}")
print(f"Test samples after filter: {len(filtered_test)}")

assert train_classes == test_classes, "Train/test class mismatch after filtering"

batch_size = BATCH_SIZE
N_CLUSTERS_MSCOCO_IMAGENET = len(train_classes)

train_loader = DataLoader(
    filtered_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=mscoco_imagenet_collate_fn,
)

test_loader = DataLoader(
    filtered_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=mscoco_imagenet_collate_fn,
)

[Loaded COCO ImageNet] 118287 samples from /mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/ViT-L-16-SigLIP2-256___webli/precomputed_train2017_clip_imagenet | vision_emb shape=(118287, 1024)
[Loaded COCO ImageNet] 5000 samples from /mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/ViT-L-16-SigLIP2-256___webli/precomputed_val2017_clip_imagenet | vision_emb shape=(5000, 1024)
Kept classes (train >= 10 & in test): 517
Train classes after filter: 517
Test classes after filter: 517
Train samples after filter: 113777
Test samples after filter: 4963


In [10]:
# %% [code]
def get_dims_gap_imagenet(loader, max_samples=20_000, device=None):
    """
    Compute normalized per-dimension gap:
        |mean(text_dim) - mean(vision_dim)|
    over a subset of samples.
    """
    if device is None:
        device = globals().get("device", "cpu")

    x_sum = None
    y_sum = None
    seen = 0

    with torch.no_grad():
        for text_b, vis_b, _ in tqdm(loader, desc="Compute per-dim gap"):
            X = F.normalize(text_b.to(device), dim=-1)
            Y = F.normalize(vis_b.to(device), dim=-1)

            b = X.shape[0]
            if x_sum is None:
                x_sum = X.sum(dim=0)
                y_sum = Y.sum(dim=0)
            else:
                x_sum += X.sum(dim=0)
                y_sum += Y.sum(dim=0)

            seen += b
            if seen >= max_samples:
                break

    mean_x = x_sum / max(seen, 1)
    mean_y = y_sum / max(seen, 1)

    gap_dim = (mean_x - mean_y).abs().detach().cpu().numpy()
    gap_dim = gap_dim / (gap_dim.max() + 1e-12)

    top_gap_idx = np.argsort(-gap_dim)
    return gap_dim, top_gap_idx

### Gap-per-dimension and subspace importance
Here we use already what we know to be the best d_sub for model-dataset combination

In [11]:
sample_text_emb, sample_img_emb, _ = filtered_train[0]
embedding_dim = int(sample_img_emb.shape[-1])
d_sub = embedding_dim if D_SUB is None else int(D_SUB)

if d_sub > embedding_dim:
    raise ValueError(f"d_sub={d_sub} cannot be larger than embedding_dim={embedding_dim}")

sub_model = fit_subspace_alignment(
    train_loader,
    n_fit=N_FIT_SAMPLES,
    d_sub=d_sub,
    device=device,
)

important_dims_txt, important_dims_img, important_joint_dims = analyze_subspace_dimensions(
    sub_model,
    device=device,
)

gap_dims, top_gap_idx = get_dims_gap_imagenet(
    train_loader,
    max_samples=GAP_MAX_SAMPLES,
    device=device,
)

print(f"MODEL_NAME={MODEL_NAME} | embedding_dim={embedding_dim} | d_sub={d_sub}")

Collected 100000 samples of dimension 1024, these will be used to fit the subspace alignment model with d_sub=512.


Compute per-dim gap:  88%|████████▊ | 390/445 [00:01<00:00, 354.30it/s]

MODEL_NAME=ViT-L-16-SigLIP2-256___webli | embedding_dim=1024 | d_sub=512


In [12]:
first_K = 5
order_gap = np.argsort(-gap_dims)
order_imp = np.argsort(-important_joint_dims)

ordered_gap_values = gap_dims[order_gap]
ordered_imp_values = important_joint_dims[order_imp]

top_gap_dims = order_gap[:first_K]
top_imp_dims = order_imp[:first_K]

In [13]:
print(f"First {first_K} per gap ==>", top_gap_dims)
print(f"First {first_K} gap values ==>", ordered_gap_values[:first_K])
print(f"First {first_K} per importance ==>", top_imp_dims)
print(f"First {first_K} importance values ==>", ordered_imp_values[:first_K])

First 5 per gap ==> [ 242  449 1001   78  205]
First 5 gap values ==> [1.         0.8215908  0.46706682 0.413949   0.37543797]
First 5 per importance ==> [575 715 506 156 176]
First 5 importance values ==> [1.         0.97992979 0.96611139 0.96562513 0.94899053]


### SAMPLE AND VIZ

In [ ]:
import json
import open_clip
from PIL import Image

# -------------------------------
# 1) Build the configured OpenCLIP model
# -------------------------------
viz_device = torch.device(device)
if viz_device.type == "cuda":
    torch.cuda.set_device(viz_device)

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL,
    pretrained=CLIP_PRETRAINED,
)
model = model.to(viz_device).eval()

export_root = HEATMAP_EXPORT_ROOT / DATASET_NAME / MODEL_NAME / str(d_sub)
export_root.mkdir(parents=True, exist_ok=True)

train_image_dir = DATASET_ROOT / "train2017"
val_image_dir = DATASET_ROOT / "val2017"
caption_cache = {}

def resolve_image_path(sample_filename):
    for split_name, split_dir in (("train2017", train_image_dir), ("val2017", val_image_dir)):
        image_path = split_dir / sample_filename
        if image_path.exists():
            return split_name, image_path
    raise FileNotFoundError(f"Could not find {sample_filename} in train2017 or val2017")

def load_captions_for_split(split_name):
    if split_name not in caption_cache:
        annotations_path = DATASET_ROOT / "annotations" / f"captions_{split_name}.json"
        if not annotations_path.exists():
            raise FileNotFoundError(f"Could not find annotations file: {annotations_path}")
        with open(annotations_path, "r") as f:
            payload = json.load(f)
        captions_by_image = {}
        for ann in payload["annotations"]:
            image_id = int(ann["image_id"])
            captions_by_image.setdefault(image_id, []).append(ann["caption"])
        caption_cache[split_name] = {
            "annotations_path": annotations_path,
            "captions_by_image": captions_by_image,
        }
    return caption_cache[split_name]

def extract_visual_bundle(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image_pil).unsqueeze(0).to(viz_device)

    with torch.no_grad():
        visual = model.visual
        x = visual._embeds(image_tensor)
        x = visual.transformer(x)
        x = visual.ln_post(x)

        cls_token = x[:, 0, :]
        spatial_tokens = x[:, 1:, :]

        pooled, _ = visual._pool(x)
        if visual.proj is not None:
            if isinstance(visual.proj, torch.nn.Linear):
                image_features = visual.proj(pooled)
            else:
                image_features = pooled @ visual.proj
        else:
            image_features = pooled

    n_patch_tokens = int(spatial_tokens.shape[1])
    grid_size = int(np.sqrt(n_patch_tokens))
    if grid_size * grid_size != n_patch_tokens:
        raise ValueError(
            f"Expected a square patch grid, but got n_patch_tokens={n_patch_tokens} for model {MODEL_NAME}."
        )

    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=viz_device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=viz_device).view(1, 3, 1, 1)
    img_vis = image_tensor * std + mean
    img_vis = img_vis[0].detach().cpu().permute(1, 2, 0).numpy()
    img_vis = np.clip(img_vis, 0, 1)

    return {
        "image_path": image_path,
        "image_tensor": image_tensor,
        "image_features": image_features,
        "cls_token": cls_token,
        "spatial_tokens": spatial_tokens,
        "img_vis": img_vis,
        "n_patch_tokens": n_patch_tokens,
        "grid_size": grid_size,
    }

def build_dim_heatmap(bundle, dim_idx):
    heat = bundle["spatial_tokens"][0, :, dim_idx].reshape(1, 1, bundle["grid_size"], bundle["grid_size"])
    heat = F.interpolate(
        heat,
        size=bundle["image_tensor"].shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    heat = heat[0, 0].detach().cpu()
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    return heat.numpy()

def aggregate_heatmap_v2(bundle, dim_indices):
    heatmaps = [build_dim_heatmap(bundle, int(dim_idx)) for dim_idx in dim_indices]
    agg = np.mean(np.stack(heatmaps, axis=0), axis=0)
    agg = (agg - agg.min()) / (agg.max() - agg.min() + 1e-8)
    return agg

def save_two_row_figure(bundle, sample_dir):
    fig, axes = plt.subplots(2, first_K, figsize=(4 * first_K, 8))
    axes = np.array(axes, dtype=object).reshape(2, first_K)

    for col, dim_idx in enumerate(top_gap_dims):
        ax = axes[0, col]
        heat = build_dim_heatmap(bundle, int(dim_idx))
        ax.imshow(bundle["img_vis"])
        ax.imshow(heat, cmap="jet", alpha=0.5)
        ax.axis("off")
        ax.set_title(f"gap dim={int(dim_idx)}\nscore={ordered_gap_values[col]:.3f}")

    for col, dim_idx in enumerate(top_imp_dims):
        ax = axes[1, col]
        heat = build_dim_heatmap(bundle, int(dim_idx))
        ax.imshow(bundle["img_vis"])
        ax.imshow(heat, cmap="jet", alpha=0.5)
        ax.axis("off")
        ax.set_title(f"importance dim={int(dim_idx)}\nscore={ordered_imp_values[col]:.3f}")

    axes[0, 0].set_ylabel("Top by gap", fontsize=12)
    axes[1, 0].set_ylabel("Top by importance", fontsize=12)
    fig.suptitle(f"First {first_K} heatmap images by gap vs importance", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    output_path = sample_dir / "two_row.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return output_path

def save_aggregate_figure(bundle, sample_dir):
    gap_heat_avg = aggregate_heatmap_v2(bundle, top_gap_dims)
    imp_heat_avg = aggregate_heatmap_v2(bundle, top_imp_dims)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(bundle["img_vis"])
    axes[0].imshow(gap_heat_avg, cmap="jet", alpha=0.5)
    axes[0].axis("off")
    axes[0].set_title(
        f"Gap aggregate heatmap (average)\nTop-{first_K} dims: {[int(d) for d in top_gap_dims]}"
    )

    axes[1].imshow(bundle["img_vis"])
    axes[1].imshow(imp_heat_avg, cmap="jet", alpha=0.5)
    axes[1].axis("off")
    axes[1].set_title(
        f"Importance aggregate heatmap (average)\nTop-{first_K} dims: {[int(d) for d in top_imp_dims]}"
    )

    fig.suptitle("Averaged top-K heatmaps: gap vs importance", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    output_path = sample_dir / "aggregate_average.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return output_path

def export_sample_heatmaps(sample_filename):
    resolved_split, image_path = resolve_image_path(sample_filename)
    caption_info = load_captions_for_split(resolved_split)
    image_id = int(Path(sample_filename).stem)
    captions = caption_info["captions_by_image"].get(image_id, [])
    if not captions:
        raise ValueError(f"No captions found for image_id={image_id} in split {resolved_split}")

    bundle = extract_visual_bundle(image_path)
    sample_dir = export_root / f"{resolved_split}_{Path(sample_filename).stem}"
    sample_dir.mkdir(parents=True, exist_ok=True)

    two_row_path = save_two_row_figure(bundle, sample_dir)
    aggregate_path = save_aggregate_figure(bundle, sample_dir)

    metadata = {
        "dataset": DATASET_NAME,
        "model_name": MODEL_NAME,
        "clip_model": CLIP_MODEL,
        "clip_pretrained": CLIP_PRETRAINED,
        "d_sub": int(d_sub),
        "embedding_dim": int(bundle["image_features"].shape[-1]),
        "sample_filename": sample_filename,
        "image_id": image_id,
        "resolved_split": resolved_split,
        "image_path": str(image_path),
        "captions_path": str(caption_info["annotations_path"]),
        "captions": captions,
        "first_K": int(first_K),
        "top_gap_dims": [int(d) for d in top_gap_dims],
        "top_gap_scores": [float(v) for v in ordered_gap_values[:first_K]],
        "top_importance_dims": [int(d) for d in top_imp_dims],
        "top_importance_scores": [float(v) for v in ordered_imp_values[:first_K]],
        "n_patch_tokens": int(bundle["n_patch_tokens"]),
        "patch_grid_size": int(bundle["grid_size"]),
        "spatial_token_dim": int(bundle["spatial_tokens"].shape[-1]),
        "export_dir": str(sample_dir),
        "two_row_figure": str(two_row_path),
        "aggregate_figure": str(aggregate_path),
    }

    metadata_path = sample_dir / "metadata.json"
    metadata["metadata_json"] = str(metadata_path)
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return metadata

export_records = []
for sample_filename in SAMPLE_IMAGE_FILENAMES:
    metadata = export_sample_heatmaps(sample_filename)
    export_records.append(metadata)

print(f"Exported {len(export_records)} samples to {export_root}")
for item in export_records:
    print(f"- {item['sample_filename']} [{item['resolved_split']}] -> {item['export_dir']}")


AttributeError: 'TimmModel' object has no attribute '_embeds'

In [ ]:
print(f"Export root: {export_root}")
print(f"Total exported samples: {len(export_records)}")
for item in export_records:
    print()
    print(f"sample          : {item['sample_filename']}")
    print(f"resolved_split  : {item['resolved_split']}")
    print(f"two_row_figure  : {item['two_row_figure']}")
    print(f"aggregate_figure: {item['aggregate_figure']}")
    print(f"metadata_json   : {item['metadata_json']}")


Export root: /mnt/media/emanuele/few_dimensions/figures_paper/heatmaps/mscoco_imagenet_labels/ViT-B-16___laion2b_s34b_b88k/512
Total exported samples: 11

sample          : 000000022892.jpg
resolved_split  : val2017
two_row_figure  : /mnt/media/emanuele/few_dimensions/figures_paper/heatmaps/mscoco_imagenet_labels/ViT-B-16___laion2b_s34b_b88k/512/val2017_000000022892/two_row.png
aggregate_figure: /mnt/media/emanuele/few_dimensions/figures_paper/heatmaps/mscoco_imagenet_labels/ViT-B-16___laion2b_s34b_b88k/512/val2017_000000022892/aggregate_average.png
metadata_json   : /mnt/media/emanuele/few_dimensions/figures_paper/heatmaps/mscoco_imagenet_labels/ViT-B-16___laion2b_s34b_b88k/512/val2017_000000022892/metadata.json

sample          : 000000001761.jpg
resolved_split  : val2017
two_row_figure  : /mnt/media/emanuele/few_dimensions/figures_paper/heatmaps/mscoco_imagenet_labels/ViT-B-16___laion2b_s34b_b88k/512/val2017_000000001761/two_row.png
aggregate_figure: /mnt/media/emanuele/few_dimensio

In [ ]:
export_records

[{'dataset': 'mscoco_imagenet_labels',
  'model_name': 'ViT-B-16___laion2b_s34b_b88k',
  'clip_model': 'ViT-B-16',
  'clip_pretrained': 'laion2b_s34b_b88k',
  'd_sub': 512,
  'embedding_dim': 512,
  'sample_filename': '000000022892.jpg',
  'image_id': 22892,
  'resolved_split': 'val2017',
  'image_path': '/mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/val2017/000000022892.jpg',
  'captions_path': '/mnt/media/emanuele/few_dimensions/dataset/mscoco/data/mscoco/annotations/captions_val2017.json',
  'captions': ['A cats sits on the table with a dog outside.',
   'A dog and a cat are watching the plants on the table..',
   'A dog looks through a window at a cat sitting on a table. ',
   'A cat sitting on a table next to plants and a dog outside of a sliding glass door, looking in.',
   'Dog looking in a window at a cat next to plants.'],
  'first_K': 5,
  'top_gap_dims': [230, 421, 160, 348, 150],
  'top_gap_scores': [1.0,
   0.8615530729293823,
   0.4288294017314911,
   0.40